In [14]:
import os, warnings, itertools
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
np.random.seed(42)

DATA_PATH = 'data/food_waste_features.csv'
MODEL_DIR = 'deployment_models'
os.makedirs(MODEL_DIR, exist_ok=True)

# -----------------------------
# Load & preprocess
# -----------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
os.chdir("/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence")

df = pd.read_csv(DATA_PATH, parse_dates=['time_bin'])

# Aggregate daily
daily = df.groupby([pd.Grouper(key='time_bin', freq='D'), 'Canteen_Section']).agg(
    waste_total=('Waste_Weight_kg', 'sum'),
    foot_traffic=('Foot_Traffic', 'sum'),
    is_holiday=('is_holiday', 'max'),
    is_special_day=('is_special_day', 'max')
).reset_index().rename(columns={'time_bin': 'ds', 'Canteen_Section': 'section'})

# Fill missing days
all_dates = pd.date_range(daily['ds'].min(), daily['ds'].max(), freq='D')
all_sections = daily['section'].unique()
daily_full = pd.DataFrame(list(itertools.product(all_dates, all_sections)),
                          columns=['ds', 'section'])
daily = daily_full.merge(daily, on=['ds', 'section'], how='left').fillna(0)

# Clip extreme 1% values
for sec in all_sections:
    q99 = daily.loc[daily['section'] == sec, 'waste_total'].quantile(0.99)
    daily.loc[daily['section'] == sec, 'waste_total'] = daily.loc[daily['section'] == sec, 'waste_total'].clip(upper=q99)

daily['y'] = daily['waste_total']  # RAW target

# -----------------------------
# Split dates
# -----------------------------
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15

all_dates = sorted(daily['ds'].unique())
n = len(all_dates)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)

train_end = all_dates[n_train-1]
val_end   = all_dates[n_train+n_val-1]
test_start = all_dates[n_train+n_val]

def split_section(df_sec):
    train = df_sec[df_sec['ds'] <= train_end]
    val = df_sec[(df_sec['ds'] > train_end) & (df_sec['ds'] <= val_end)]
    test = df_sec[df_sec['ds'] > val_end]
    return train, val, test

# -----------------------------
# Metrics
# -----------------------------
def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom > 0
    return np.mean(2 * np.abs(y_pred[mask]-y_true[mask])/denom[mask])*100 if mask.any() else np.nan

def metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return dict(
        RMSE=np.sqrt(mean_squared_error(y_true, y_pred)),
        MAE=mean_absolute_error(y_true, y_pred),
        sMAPE=smape(y_true, y_pred),
        R2=r2_score(y_true, y_pred)
    )

FEAT_COLS = ['ds','y','foot_traffic','is_holiday','is_special_day']

def build_prophet(params):
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=params['changepoint_prior_scale'],
        seasonality_prior_scale=params['seasonality_prior_scale'],
        seasonality_mode=params['seasonality_mode'],
        n_changepoints=50,
        changepoint_range=0.9
    )
    m.add_regressor('foot_traffic', prior_scale=params['regressor_prior'])
    m.add_regressor('is_holiday', prior_scale=params['regressor_prior'])
    m.add_regressor('is_special_day', prior_scale=params['regressor_prior'])
    m.add_seasonality(name='monthly', period=30.5, fourier_order=5)
    return m

def evaluate_model(params, df_sec):
    train_df, val_df, test_df = split_section(df_sec)
    trainval_df = pd.concat([train_df, val_df])
    m = build_prophet(params)
    m.fit(trainval_df[FEAT_COLS])
    future = test_df[FEAT_COLS].drop(columns=['y'])
    forecast = m.predict(future)
    preds = forecast.set_index('ds')['yhat'].loc[test_df['ds']]
    y_true = test_df['y'].values
    y_pred = preds.values.clip(min=0)
    return metrics(y_true, y_pred)

# -----------------------------
# Grid search hyperparameters
# -----------------------------
sections = sorted(daily['section'].unique())
final_metrics = []

cps_options = [0.05,0.1,0.2]
season_options = ['additive','multiplicative']
reg_prior_options = [0.5,1.0,1.5]

best_params_list = []

for sec in sections:
    print(f'\n===== Section {sec} =====')
    df_sec = daily[daily['section']==sec].sort_values('ds')

    # Base metrics
    base_params = dict(changepoint_prior_scale=0.1, seasonality_prior_scale=5.0, seasonality_mode='additive', regressor_prior=0.5)
    base_metrics = evaluate_model(base_params, df_sec)

    # Grid search for R2
    best_r2 = -np.inf
    best_params = base_params.copy()
    for cps in cps_options:
        for season in season_options:
            for rp in reg_prior_options:
                candidate = dict(changepoint_prior_scale=cps, seasonality_prior_scale=5.0, seasonality_mode=season, regressor_prior=rp)
                test_metrics = evaluate_model(candidate, df_sec)
                if test_metrics['R2'] > best_r2:
                    best_r2 = test_metrics['R2']
                    best_params = candidate
                    best_test_metrics = test_metrics

    print(f'Base R2={base_metrics["R2"]:.3f}, Tuned R2={best_r2:.3f}')

    # Save best params for this section
    best_params_list.append({'Section': sec, **best_params})

    final_metrics.append({
        'Section': sec,
        'Base_RMSE': base_metrics['RMSE'], 'Base_MAE': base_metrics['MAE'], 'Base_sMAPE': base_metrics['sMAPE'], 'Base_R2': base_metrics['R2'],
        'Tuned_RMSE': best_test_metrics['RMSE'], 'Tuned_MAE': best_test_metrics['MAE'], 'Tuned_sMAPE': best_test_metrics['sMAPE'], 'Tuned_R2': best_test_metrics['R2']
    })

# Convert to DataFrame and save
best_params_df = pd.DataFrame(best_params_list).set_index('Section')
best_params_df.to_csv(f'{MODEL_DIR}/best_hyperparameters_Prophet.csv')
print("\nBest hyperparameters per section:")
print(best_params_df)

Mounted at /content/drive

===== Section A =====
Base R2=0.896, Tuned R2=0.896

===== Section B =====
Base R2=0.887, Tuned R2=0.897

===== Section C =====
Base R2=0.883, Tuned R2=0.886

===== Section D =====
Base R2=0.861, Tuned R2=0.863

Best hyperparameters per section:
         changepoint_prior_scale  seasonality_prior_scale seasonality_mode  \
Section                                                                      
A                           0.20                      5.0         additive   
B                           0.10                      5.0   multiplicative   
C                           0.05                      5.0         additive   
D                           0.20                      5.0         additive   

         regressor_prior  
Section                   
A                    0.5  
B                    1.5  
C                    1.5  
D                    1.0  
